In [1]:
import pipeline2 as pipeline
import model_trainer

# 1. Fetch clean, stationary data (Force refresh to ensure we get the new Term Spread)
df_raw = pipeline.fetch_and_prepare_data(force_refresh=True)

# 2. Apply the dynamic rolling thresholds and target smoothing
# Using the quantiles you identified in your visual EDA
df_regimes = pipeline.create_multiclass_target(
    df=df_raw, 
    lower_quant=0.86, 
    upper_quant=0.95, 
    window=252,
    shock_smooth=3,
    elevated_smooth=4
)

# 3. Define the features we want to lag
# We include returns, volatilities, and our stationary macro shocks
features_to_lag = [
    'Log_Return', 'Sq_Log_Return', 'Vol_GARCH', 'Vol_EGARCH', 
    'VIX_Change', 'Oil_Change', 'CPI_MoM', 'FedFunds_Diff', 'Term_Spread_Diff'
]

# 4. Engineer lag features
# Now the model can see how inflation changed 5 days ago, or VIX 10 days ago
df_model_ready = model_trainer.engineer_lag_features(df_regimes, feature_cols=features_to_lag)

# 5. Train and evaluate
model, feature_names, X_test, y_test = model_trainer.train_and_evaluate_xgboost(
    df=df_model_ready, 
    target_col='Target_Smooth_10d'
)

# 6. Check the feature importance
model_trainer.plot_feature_importance(model, feature_names)

Starting data pipeline...


ValueError: Too Many Requests.  Exceeded Rate Limit